In [ ]:
import numpy as np
import pickle 

from sbi.utils import BoxUniform
from scipy.integrate import odeint
import torch

_ = torch.manual_seed(0)
_ = np.random.seed(0)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)                                                                                                                                                                                                                                                                                                                               

/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


Using device: cpu


In [2]:
def ode_system(y, t, beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N):
    S, E, I, H, R, D = y
    dSdt = - (beta_I * I + beta_H * H) * S / N
    dEdt = (beta_I * I + beta_H * H) * S / N - kappa * E
    dIdt =  kappa * E - (alpha + gamma_I + delta_I) * I
    dHdt = alpha * I - (gamma_H + delta_H) * H
    dRdt = gamma_I * I + gamma_H * H
    dDdt = delta_I * I + delta_H *H
    return [dSdt, dEdt, dIdt, dHdt, dRdt, dDdt]

def simulate(parameters, ic, T):
    beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H = parameters
    S0, E0, I0, H0, R0, D0 = ic

    N = S0 + E0 + I0 + H0 + R0 + D0
    y0=ic
    
    t = np.arange(T+1)
    ret = odeint(ode_system, y0, t, args=(beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N))

    cases1 = kappa * ret[1:,1]
    cases2 = alpha * ret[1:,2]
    cases3 = delta_I * ret[1:,2] + delta_H * ret[1:,3]
    
    return np.stack([cases1, cases2, cases3], axis=1)

In [4]:
N = 100000
E0 = 0
I0 = 10
H0 = 0
R0 = 0
D0 = 0
S0 = N - E0 - I0 - H0 - R0 - D0

duration=100

init_cond = [S0, E0, I0, H0, R0, D0]

In [5]:
low  = torch.tensor([0.7, 0.2, 0.1, 0.1, 0.1, 0.001, 0.1, 0.001])
high = torch.tensor([1.3, 0.8, 0.3, 0.3, 0.3, 0.05, 0.4, 0.05])

prior = BoxUniform(low=low, high=high)

In [6]:
num_simulations = 100_000     
thetas = prior.sample((num_simulations,))

In [7]:
xs = []
for theta in thetas:
    x_sim = simulate(theta.numpy(), init_cond, duration)
    xs.append(x_sim)

xs = torch.tensor(xs, dtype=torch.float32)

/tmp/ipykernel_4053003/1705214159.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  xs = torch.tensor(xs, dtype=torch.float32)


In [ ]:
with open("./M2_dataset.pkl", "wb") as handle:
    pickle.dump(xs, handle,protocol=pickle.HIGHEST_PROTOCOL)

with open("./M2_params.pkl", "wb") as handle:
    pickle.dump(thetas, handle)